In [2]:
import numpy as np
def make_mean_zero(x):
    """Make column means zero."""
    return x - x.mean(axis=0)


def make_std_one(x):
    """Make column standard deviations one."""
    return x / x.std(axis=0, ddof=1)


def varcov(x):
    """
    Compute sample covariance matrix S and estimated variance of each entry VS.
    
    Returns:
        S  : (p, p) sample covariance matrix
        VS : (p, p) estimated variance of each entry in S
    """
    n, p = x.shape
    xc = make_mean_zero(x)
    S = np.cov(xc, rowvar=False)  # (p, p)

    # XC1[i, j, k] = xc[k, i],  XC2[i, j, k] = xc[k, j]
    XC1 = np.tile(xc.T[:, np.newaxis, :], (1, p, 1))  # (p, p, n)
    XC2 = np.tile(xc.T[np.newaxis, :, :], (p, 1, 1))  # (p, p, n)
    VS = np.var(XC1 * XC2, axis=2, ddof=1) * n / (n - 1) ** 2
    return S, VS


def corshrink(x):
    """
    Shrinkage estimate of a correlation matrix.
    Equation on p.4 of Schafer & Strimmer (2005).

    Args:
        x : (n, p) data matrix

    Returns:
        Rhat   : (p, p) shrunk correlation matrix
        lambda_ : shrinkage coefficient
    """
    n, p = x.shape
    sx = make_mean_zero(x)
    sx = make_std_one(sx)
    r, vr = varcov(sx)

    # Sum over lower triangle (off-diagonal)
    tril_idx = np.tril_indices(p, k=-1)
    offdiag_sum_rij2 = np.sum(r[tril_idx] ** 2)
    offdiag_sum_vrij = np.sum(vr[tril_idx])

    lambda_ = offdiag_sum_vrij / offdiag_sum_rij2
    lambda_ = float(np.clip(lambda_, 0, 1))

    Rhat = (1 - lambda_) * r
    np.fill_diagonal(Rhat, 1.0)
    return Rhat, lambda_


def varshrink(x):
    """
    Shrinkage estimate of variances.
    Equations 10 and 11 of Opgen-Rhein & Strimmer (2007).

    Args:
        x : (n, p) data matrix

    Returns:
        sv     : (p,) shrunk variances
        lambda_ : shrinkage coefficient
    """
    S, VS = varcov(x)
    v = np.diag(S)
    vv = np.diag(VS)

    vtarget = np.median(v)
    numerator = np.sum(vv)
    denominator = np.sum((v - vtarget) ** 2)

    lambda_ = float(np.clip(numerator / denominator, 0, 1))
    sv = (1 - lambda_) * v + lambda_ * vtarget
    return sv, lambda_


def covshrink_kpm(x, shrinkvar=False):
    """
    Shrinkage estimate of a covariance matrix using optimal shrinkage coefficient.
    Based on Schaefer & Strimmer (2005), vectorized and simplified by Kevin Murphy.

    Args:
        x         : (n, p) data matrix
        shrinkvar : if True, also shrink the diagonal variance terms (default False)

    Returns:
        s      : (p, p) positive definite covariance matrix
        lamcor : shrinkage coefficient for the correlation matrix
        lamvar : shrinkage coefficient for the variances (0 if shrinkvar=False)
    """
    n, p = x.shape

    # Trivial 1D case
    if p == 1:
        return np.var(x, ddof=1), 0.0, 0.0

    if shrinkvar:
        v, lamvar = varshrink(x)
    else:
        v = np.var(x, axis=0, ddof=1)
        lamvar = 0.0
    dsv = np.diag(np.sqrt(v))        # (p, p) diagonal matrix of std devs
    r, lamcor = corshrink(x)         # shrunk correlation matrix

    s = dsv @ r @ dsv                # convert correlation → covariance
    return s, lamcor, lamvar


# ── Quick sanity check ────────────────────────────────────────────────────────
if __name__ == "__main__":
    rng = np.random.default_rng(42)
    X = rng.standard_normal((50, 10))

    S, lamcor, lamvar = covshrink_kpm(X, shrinkvar=False)
    print("Covariance matrix shape:", S.shape)
    print(f"lamcor = {lamcor:.4f},  lamvar = {lamvar:.4f}")
    print("Is positive definite:", np.all(np.linalg.eigvalsh(S) > 0))

    S2, lamcor2, lamvar2 = covshrink_kpm(X, shrinkvar=True)
    print(f"\nWith variance shrinkage:")
    print(f"lamcor = {lamcor2:.4f},  lamvar = {lamvar2:.4f}")
    print("Is positive definite:", np.all(np.linalg.eigvalsh(S2) > 0))

Covariance matrix shape: (10, 10)
lamcor = 0.9079,  lamvar = 0.0000
Is positive definite: True

With variance shrinkage:
lamcor = 0.9079,  lamvar = 1.0000
Is positive definite: True


In [ ]:
import warnings
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import networkx as nx
from pathlib import Path

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these paths before running
# ─────────────────────────────────────────────────────────────────────────────
FMRI_PATH  = "TASK1Data/tractor/functional/data.nii.gz"          # 4-D fMRI NIfTI
ATLAS_PATH = "TASK1Data/tractor/functional/parcellation.nii.gz"         # 3-D integer-labelled parcellation
                                     # (must be in same space/resolution)
OUTPUT_DIR = Path("Task1_outputs")  # directory to save figures and results
OUTPUT_DIR.mkdir(exist_ok=True)

LAMBDAS         = np.round(np.arange(0.1, 0.85, 0.1), 2)   # 0.1 … 0.8
THRESHOLD       = 0.1        # absolute correlation threshold
KEEP_NEGATIVE   = True       # set False to discard negative edges
# ─────────────────────────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════════════════
#  1.  TIME-SERIES EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_roi_timeseries(fmri_path: str, atlas_path: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns
    -------
    ts     : (n_timepoints, n_regions) array of mean BOLD signals
    labels : (n_regions,) integer region labels
    """
    print("Loading NIfTI files …")
    fmri_img  = nib.load(fmri_path)
    atlas_img = nib.load(atlas_path)

    fmri_data  = fmri_img.get_fdata()   # (X, Y, Z, T)
    atlas_data = atlas_img.get_fdata().astype(int)  # (X, Y, Z)

    if fmri_data.shape[:3] != atlas_data.shape:
        raise ValueError(
            f"fMRI spatial dims {fmri_data.shape[:3]} ≠ atlas dims {atlas_data.shape}. "
            "Resample to the same space first."
        )

    labels = np.unique(atlas_data)
    labels = labels[labels > 0]          # drop background (0)

    n_t = fmri_data.shape[3]
    ts  = np.zeros((n_t, len(labels)), dtype=np.float64)

    for idx, lbl in enumerate(labels):
        mask       = atlas_data == lbl
        ts[:, idx] = fmri_data[mask].mean(axis=0)

    print(f"  → {len(labels)} regions, {n_t} time points")
    return ts, labels


# ══════════════════════════════════════════════════════════════════════════════
#  2.  SCHÄFER & STRIMMER CORSHRINK  (manual lambda)
# ══════════════════════════════════════════════════════════════════════════════

def make_mean_zero(x: np.ndarray) -> np.ndarray:
    return x - x.mean(axis=0)


def make_std_one(x: np.ndarray) -> np.ndarray:
    return x / x.std(axis=0, ddof=1)


def corshrink_manual(x: np.ndarray, lam: float) -> np.ndarray:
    """
    Shrinkage correlation matrix with a *fixed* lambda.

    The target is the identity (zero off-diagonal correlations).
    Rhat = (1 - lam) * R_sample  +  lam * I

    Parameters
    ----------
    x   : (n, p) data matrix
    lam : shrinkage parameter in [0, 1]
              0 → pure sample correlation matrix
              1 → identity (no correlation)

    Returns
    -------
    Rhat : (p, p) shrunk correlation matrix
    """
    p  = x.shape[1]
    sx = make_mean_zero(x)
    sx = make_std_one(sx)

    # Sample correlation matrix
    xc = make_mean_zero(sx)
    n  = xc.shape[0]
    R  = (xc.T @ xc) / (n - 1)

    # Shrink off-diagonal toward zero
    Rhat = (1 - lam) * R
    np.fill_diagonal(Rhat, 1.0)
    return Rhat


# ══════════════════════════════════════════════════════════════════════════════
#  3.  THRESHOLDING & BINARISATION
# ══════════════════════════════════════════════════════════════════════════════

def threshold_and_binarise(R: np.ndarray,
                            threshold: float = 0.1,
                            keep_negative: bool = True) -> np.ndarray:
    """
    Returns a symmetric binary adjacency matrix.

    If keep_negative=True  : edges where |r| > threshold are retained (value 1).
    If keep_negative=False : only positive edges where r > threshold are kept.
    """
    if keep_negative:
        A = (np.abs(R) > threshold).astype(float)
    else:
        A = (R > threshold).astype(float)
    np.fill_diagonal(A, 0)
    return A


# ══════════════════════════════════════════════════════════════════════════════
#  4.  GRAPH METRICS  (NetworkX / BCT-style)
# ══════════════════════════════════════════════════════════════════════════════

def compute_graph_metrics(A: np.ndarray) -> dict:
    """
    Computes BCT-inspired graph metrics on a binary undirected graph.

    Returns a dict with scalar summary statistics.
    """
    p  = A.shape[0]
    G  = nx.from_numpy_array(A)

    # ── Edge density ────────────────────────────────────────────────────────
    n_edges    = G.number_of_edges()
    max_edges  = p * (p - 1) / 2
    density    = n_edges / max_edges if max_edges > 0 else 0.0

    # ── Clustering coefficient ───────────────────────────────────────────────
    clustering = nx.average_clustering(G)

    # ── Path length & global efficiency ─────────────────────────────────────
    # Use largest connected component to avoid inf path lengths
    if nx.is_connected(G):
        Gc = G
    else:
        Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

    if len(Gc) > 1:
        avg_path_length  = nx.average_shortest_path_length(Gc)
        global_efficiency = nx.global_efficiency(G)   # defined even if disconnected
    else:
        avg_path_length  = np.nan
        global_efficiency = 0.0

    # ── Small-world coefficient σ  (requires a random reference) ────────────
    # σ = (C/C_rand) / (L/L_rand)
    # Approximate C_rand and L_rand analytically for an ER graph with same density
    if density > 0 and p > 1:
        C_rand = density                          # ER clustering ≈ p (density)
        L_rand = (np.log(p) / np.log(max(density * p, 2)))  # ER path length approx
        sigma  = (clustering / C_rand) / (avg_path_length / L_rand) if not np.isnan(avg_path_length) else np.nan
    else:
        sigma  = np.nan

    return {
        "n_edges"          : n_edges,
        "density"          : density,
        "clustering"       : clustering,
        "avg_path_length"  : avg_path_length,
        "global_efficiency": global_efficiency,
        "small_world_sigma": sigma,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  5.  VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_connectome_sweep(results: list[dict], lambdas: np.ndarray,
                          keep_negative: bool, output_dir: Path):
    """
    Two figures:
      Fig 1 — correlation matrices for each lambda
      Fig 2 — graph metrics vs lambda
    """
    n_lam = len(lambdas)
    neg_tag = "with_neg" if keep_negative else "no_neg"

    # ── Figure 1: correlation matrices ──────────────────────────────────────
    ncols = 4
    nrows = int(np.ceil(n_lam / ncols))
    fig1, axes = plt.subplots(nrows, ncols,
                               figsize=(4 * ncols, 3.5 * nrows))
    axes = axes.flatten()

    for i, res in enumerate(results):
        im = axes[i].imshow(res["R"], vmin=-1, vmax=1, cmap="RdBu_r",
                             interpolation="nearest")
        axes[i].set_title(f"λ = {lambdas[i]:.1f}", fontsize=11)
        axes[i].axis("off")
        plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig1.suptitle("Shrinkage Correlation Matrices", fontsize=14, y=1.01)
    fig1.tight_layout()
    fig1.savefig(output_dir / f"correlation_matrices_{neg_tag}.png",
                  dpi=150, bbox_inches="tight")
    plt.close(fig1)

    # ── Figure 2: graph metrics vs lambda ───────────────────────────────────
    metrics_to_plot = [
        ("density",          "Edge Density"),
        ("clustering",       "Clustering Coefficient"),
        ("global_efficiency","Global Efficiency"),
        ("avg_path_length",  "Avg Path Length (LCC)"),
    ]

    fig2, axes2 = plt.subplots(2, 2, figsize=(11, 8))
    axes2 = axes2.flatten()

    for ax, (key, label) in zip(axes2, metrics_to_plot):
        vals = [r["metrics"][key] for r in results]
        ax.plot(lambdas, vals, "o-", linewidth=2, markersize=7, color="#2c7bb6")
        ax.set_xlabel("Shrinkage parameter λ", fontsize=11)
        ax.set_ylabel(label, fontsize=11)
        ax.set_title(label, fontsize=12)
        ax.set_xticks(lambdas)
        ax.grid(True, linestyle="--", alpha=0.5)

    neg_title = "(negative edges retained)" if keep_negative else "(negative edges discarded)"
    fig2.suptitle(f"Graph Metrics vs Shrinkage Parameter  {neg_title}",
                   fontsize=13)
    fig2.tight_layout()
    fig2.savefig(output_dir / f"graph_metrics_{neg_tag}.png",
                  dpi=150, bbox_inches="tight")
    plt.close(fig2)

    print(f"  Saved figures to {output_dir}/")


def print_summary_table(results: list[dict], lambdas: np.ndarray):
    header = (f"{'λ':>5}  {'Edges':>7}  {'Density':>9}  "
              f"{'Clustering':>12}  {'PathLen':>9}  {'Efficiency':>11}")
    print("\n" + header)
    print("─" * len(header))
    for lam, res in zip(lambdas, results):
        m = res["metrics"]
        pl = f"{m['avg_path_length']:.4f}" if not np.isnan(m['avg_path_length']) else "  NaN  "
        print(f"{lam:>5.1f}  {m['n_edges']:>7d}  {m['density']:>9.4f}  "
              f"{m['clustering']:>12.4f}  {pl:>9}  {m['global_efficiency']:>11.4f}")


# ══════════════════════════════════════════════════════════════════════════════
#  6.  MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(fmri_path: str, atlas_path: str,
                 lambdas: np.ndarray = LAMBDAS,
                 threshold: float = THRESHOLD,
                 keep_negative: bool = KEEP_NEGATIVE):

    # Step 1 — extract time series
    ts, labels = extract_roi_timeseries(fmri_path, atlas_path)

    # Step 2 & 3 — sweep over lambda values
    for neg in [True, False]:
        print(f"\n{'='*60}")
        print(f"  Negative edges: {'RETAINED' if neg else 'DISCARDED'}")
        print(f"{'='*60}")

        results = []
        for lam in lambdas:
            R = corshrink_manual(ts, lam)
            A = threshold_and_binarise(R, threshold=threshold, keep_negative=neg)
            metrics = compute_graph_metrics(A)
            results.append({"lam": lam, "R": R, "A": A, "metrics": metrics})
            print(f"  λ={lam:.1f}  edges={metrics['n_edges']:4d}  "
                  f"density={metrics['density']:.3f}  "
                  f"clustering={metrics['clustering']:.3f}  "
                  f"efficiency={metrics['global_efficiency']:.3f}")

        print_summary_table(results, lambdas)
        plot_connectome_sweep(results, lambdas, neg, OUTPUT_DIR)

    print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")


# ══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    run_pipeline(FMRI_PATH, ATLAS_PATH)

Loading NIfTI files …
  → 110 regions, 15 time points

  Negative edges: RETAINED
  λ=0.1  edges=4665  density=0.778  clustering=0.789  efficiency=0.889
  λ=0.2  edges=4486  density=0.748  clustering=0.762  efficiency=0.874
  λ=0.3  edges=4256  density=0.710  clustering=0.728  efficiency=0.855
  λ=0.4  edges=3994  density=0.666  clustering=0.692  efficiency=0.833
  λ=0.5  edges=3604  density=0.601  clustering=0.641  efficiency=0.801
  λ=0.6  edges=3038  density=0.507  clustering=0.576  efficiency=0.753
  λ=0.7  edges=2261  density=0.377  clustering=0.509  efficiency=0.688
  λ=0.8  edges= 927  density=0.155  clustering=0.409  efficiency=0.526

    λ    Edges    Density    Clustering    PathLen   Efficiency
───────────────────────────────────────────────────────────────
  0.1     4665     0.7781        0.7889     1.2219       0.8891
  0.2     4486     0.7483        0.7621     1.2517       0.8741
  0.3     4256     0.7099        0.7278     1.2901       0.8550
  0.4     3994     0.6662    

In [ ]:
import warnings
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from pathlib import Path

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these paths before running
# ─────────────────────────────────────────────────────────────────────────────
FMRI_PATH  = "TASK1Data/tractor/functional/data.nii.gz"          # 4-D fMRI NIfTI
ATLAS_PATH = "TASK1Data/tractor/functional/parcellation.nii.gz"         # 3-D integer-labelled parcellation
OUTPUT_DIR = Path("Task1_outputs")  # directory to save figures and results
OUTPUT_DIR.mkdir(exist_ok=True)

LAMBDAS       = np.round(np.arange(0.1, 0.85, 0.1), 2)   # 0.1 … 0.8
THRESHOLD     = 0.1
KEEP_NEGATIVE = True    # also runs a second pass with False
# ─────────────────────────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════════════════
#  1.  TIME-SERIES EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_roi_timeseries(fmri_path: str, atlas_path: str):
    """
    Returns
    -------
    ts     : (n_timepoints, n_regions) array of mean BOLD signals
    labels : (n_regions,) integer region labels
    names  : list of auto-generated region name strings
    """
    print("Loading NIfTI files …")
    fmri_img  = nib.load(fmri_path)
    atlas_img = nib.load(atlas_path)

    fmri_data  = fmri_img.get_fdata()              # (X, Y, Z, T)
    atlas_data = atlas_img.get_fdata().astype(int)  # (X, Y, Z)

    if fmri_data.shape[:3] != atlas_data.shape:
        raise ValueError(
            f"fMRI spatial dims {fmri_data.shape[:3]} ≠ atlas dims {atlas_data.shape}. "
            "Resample to the same space first."
        )

    labels = np.unique(atlas_data)
    labels = labels[labels > 0]          # drop background (0)

    n_t   = fmri_data.shape[3]
    ts    = np.zeros((n_t, len(labels)), dtype=np.float64)
    names = [f"region_{lbl:03d}" for lbl in labels]

    for idx, lbl in enumerate(labels):
        mask       = atlas_data == lbl
        ts[:, idx] = fmri_data[mask].mean(axis=0)

    print(f"  → {len(labels)} regions, {n_t} time points")
    return ts, labels, names


# ══════════════════════════════════════════════════════════════════════════════
#  2.  CORSHRINK  (manual lambda)
# ══════════════════════════════════════════════════════════════════════════════

def make_mean_zero(x: np.ndarray) -> np.ndarray:
    return x - x.mean(axis=0)


def make_std_one(x: np.ndarray) -> np.ndarray:
    return x / x.std(axis=0, ddof=1)


def corshrink_manual(x: np.ndarray, lam: float) -> np.ndarray:
    """
    Shrinkage correlation matrix with a fixed lambda.
    Rhat = (1 - lam) * R_sample  +  lam * I
    """
    sx = make_mean_zero(x)
    sx = make_std_one(sx)
    xc = make_mean_zero(sx)
    n  = xc.shape[0]
    R  = (xc.T @ xc) / (n - 1)
    Rhat = (1 - lam) * R
    np.fill_diagonal(Rhat, 1.0)
    return Rhat


# ══════════════════════════════════════════════════════════════════════════════
#  3.  THRESHOLDING & BINARISATION
# ══════════════════════════════════════════════════════════════════════════════

def threshold_and_binarise(R: np.ndarray,
                            threshold: float = 0.1,
                            keep_negative: bool = True) -> np.ndarray:
    if keep_negative:
        A = (np.abs(R) > threshold).astype(float)
    else:
        A = (R > threshold).astype(float)
    np.fill_diagonal(A, 0)
    return A


# ══════════════════════════════════════════════════════════════════════════════
#  4.  GRAPH METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_graph_metrics(A: np.ndarray) -> dict:
    p  = A.shape[0]
    G  = nx.from_numpy_array(A)

    n_edges   = G.number_of_edges()
    max_edges = p * (p - 1) / 2
    density   = n_edges / max_edges if max_edges > 0 else 0.0
    clustering = nx.average_clustering(G)

    if nx.is_connected(G):
        Gc = G
    else:
        Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

    avg_path_length   = nx.average_shortest_path_length(Gc) if len(Gc) > 1 else np.nan
    global_efficiency = nx.global_efficiency(G)

    if density > 0 and p > 1:
        C_rand = density
        L_rand = np.log(p) / np.log(max(density * p, 2))
        sigma  = (clustering / C_rand) / (avg_path_length / L_rand) if not np.isnan(avg_path_length) else np.nan
    else:
        sigma = np.nan

    return {
        "n_edges"          : n_edges,
        "density"          : density,
        "clustering"       : clustering,
        "avg_path_length"  : avg_path_length,
        "global_efficiency": global_efficiency,
        "small_world_sigma": sigma,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  5.  CONNECTOME PLOT  (single lambda)
# ══════════════════════════════════════════════════════════════════════════════

def plot_connectome(R: np.ndarray, A: np.ndarray, names: list,
                    title: str = "Connectome",
                    layout: str = "circular",
                    threshold: float = 0.1,
                    save_path: str | None = None):
    """
    Left  : correlation heatmap
    Right : network graph — edges coloured by sign, nodes sized by degree
    """
    p     = R.shape[0]
    short = [
        n.replace("_left", "_L").replace("_right", "_R")
         .replace("_Left", "_L").replace("_Right", "_R")
        for n in names
    ]

    G      = nx.from_numpy_array(A)
    pos    = nx.circular_layout(G) if layout == "circular" else nx.spring_layout(G, seed=42, k=2/np.sqrt(p))
    edges  = list(G.edges())

    # Edge colours & widths
    cmap_edges  = plt.cm.RdBu_r
    norm_edges  = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    edge_corrs  = [R[u, v] for u, v in edges]
    edge_colors = [cmap_edges(norm_edges(c)) for c in edge_corrs]
    edge_widths = [max(0.3, abs(c) * 2.5) for c in edge_corrs]

    # Node size & colour by degree
    degrees     = dict(G.degree())
    max_deg     = max(degrees.values()) if degrees else 1
    node_sizes  = [80 + 400 * (degrees[n] / max_deg) for n in G.nodes()]
    node_colors = [plt.cm.YlOrRd(0.2 + 0.8 * degrees[n] / max_deg) for n in G.nodes()]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(title, fontsize=15, fontweight="bold", y=1.01)

    # ── Heatmap ──────────────────────────────────────────────────────────────
    ax = axes[0]
    im = ax.imshow(R, vmin=-1, vmax=1, cmap="RdBu_r", interpolation="nearest")
    ax.set_title("Correlation Matrix", fontsize=12)
    ax.set_xlabel("Region index", fontsize=10)
    ax.set_ylabel("Region index", fontsize=10)
    tick_step = max(1, p // 20)
    ticks     = np.arange(0, p, tick_step)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels([short[i] for i in ticks], rotation=90, fontsize=6)
    ax.set_yticklabels([short[i] for i in ticks], fontsize=6)
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Pearson r", fontsize=9)
    cb.ax.axhline( threshold, color="k", linewidth=1.5, linestyle="--")
    cb.ax.axhline(-threshold, color="k", linewidth=1.5, linestyle="--")

    # ── Network graph ─────────────────────────────────────────────────────────
    ax2 = axes[1]
    if edges:
        nx.draw_networkx_edges(G, pos, ax=ax2, edgelist=edges,
                               edge_color=edge_colors, width=edge_widths, alpha=0.65)
    nx.draw_networkx_nodes(G, pos, ax=ax2, node_size=node_sizes,
                           node_color=node_colors, linewidths=0.5, edgecolors="k")
    if p <= 80:
        nx.draw_networkx_labels(G, pos, labels={i: short[i] for i in range(p)},
                                ax=ax2, font_size=5 if p > 40 else 7)
    ax2.set_title(f"Network Graph  ({layout} layout)\n"
                  "Nodes sized by degree · Edges coloured by r", fontsize=11)
    ax2.axis("off")

    sm  = plt.cm.ScalarMappable(cmap=cmap_edges, norm=norm_edges)
    sm.set_array([])
    cb2 = plt.colorbar(sm, ax=ax2, fraction=0.035, pad=0.02, shrink=0.7)
    cb2.set_label("Correlation (r)", fontsize=9)

    n_pos = sum(1 for c in edge_corrs if c > 0)
    n_neg = len(edge_corrs) - n_pos
    ax2.text(0.02, 0.02,
             f"Edges: {len(edges)}\nPositive: {n_pos}  ·  Negative: {n_neg}\n"
             f"Density: {len(edges) / (p*(p-1)/2):.3f}",
             transform=ax2.transAxes, fontsize=8, verticalalignment="bottom",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {save_path}")
    else:
        plt.show()


# ══════════════════════════════════════════════════════════════════════════════
#  6.  SUMMARY PLOTS  (metrics vs lambda)
# ══════════════════════════════════════════════════════════════════════════════

def plot_metrics_sweep(results: list, lambdas: np.ndarray,
                       keep_negative: bool, output_dir: Path):
    neg_tag = "with_neg" if keep_negative else "no_neg"
    metrics_to_plot = [
        ("density",          "Edge Density"),
        ("clustering",       "Clustering Coefficient"),
        ("global_efficiency","Global Efficiency"),
        ("avg_path_length",  "Avg Path Length (LCC)"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.flatten()
    for ax, (key, label) in zip(axes, metrics_to_plot):
        vals = [r["metrics"][key] for r in results]
        ax.plot(lambdas, vals, "o-", linewidth=2, markersize=7, color="#2c7bb6")
        ax.set_xlabel("Shrinkage parameter λ", fontsize=11)
        ax.set_ylabel(label, fontsize=11)
        ax.set_title(label, fontsize=12)
        ax.set_xticks(lambdas)
        ax.grid(True, linestyle="--", alpha=0.5)
    neg_title = "(negative edges retained)" if keep_negative else "(negative edges discarded)"
    fig.suptitle(f"Graph Metrics vs λ  {neg_title}", fontsize=13)
    fig.tight_layout()
    path = output_dir / f"graph_metrics_{neg_tag}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def print_summary_table(results: list, lambdas: np.ndarray):
    header = (f"{'λ':>5}  {'Edges':>7}  {'Density':>9}  "
              f"{'Clustering':>12}  {'PathLen':>9}  {'Efficiency':>11}")
    print("\n" + header)
    print("─" * len(header))
    for lam, res in zip(lambdas, results):
        m  = res["metrics"]
        pl = f"{m['avg_path_length']:.4f}" if not np.isnan(m['avg_path_length']) else "    NaN"
        print(f"{lam:>5.1f}  {m['n_edges']:>7d}  {m['density']:>9.4f}  "
              f"{m['clustering']:>12.4f}  {pl:>9}  {m['global_efficiency']:>11.4f}")


# ══════════════════════════════════════════════════════════════════════════════
#  7.  MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(fmri_path: str, atlas_path: str,
                 lambdas: np.ndarray = LAMBDAS,
                 threshold: float    = THRESHOLD):

    ts, labels, names = extract_roi_timeseries(fmri_path, atlas_path)

    for keep_neg in [True, False]:
        neg_tag   = "with_neg" if keep_neg else "no_neg"
        neg_label = "RETAINED" if keep_neg else "DISCARDED"
        print(f"\n{'='*60}")
        print(f"  Negative edges: {neg_label}")
        print(f"{'='*60}")

        results = []
        for lam in lambdas:
            R       = corshrink_manual(ts, lam)
            A       = threshold_and_binarise(R, threshold=threshold, keep_negative=keep_neg)
            metrics = compute_graph_metrics(A)
            results.append({"lam": lam, "R": R, "A": A, "metrics": metrics})

            print(f"  λ={lam:.1f}  edges={metrics['n_edges']:4d}  "
                  f"density={metrics['density']:.3f}  "
                  f"clustering={metrics['clustering']:.3f}  "
                  f"efficiency={metrics['global_efficiency']:.3f}")

            # ── Connectome plot for this lambda ──────────────────────────────
            plot_connectome(
                R, A, names,
                title     = f"Functional Connectome  (λ = {lam:.1f}, {neg_label.lower()} negatives)",
                layout    = "circular",
                threshold = threshold,
                save_path = str(OUTPUT_DIR / f"connectome_lambda_{lam:.1f}_{neg_tag}.png"),
            )

        print_summary_table(results, lambdas)
        plot_metrics_sweep(results, lambdas, keep_neg, OUTPUT_DIR)

    print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    run_pipeline(FMRI_PATH, ATLAS_PATH)
